# 05 (Kaggle) — Generation diagnosis: quantized vs full-precision inference

Rebuilt clean after a live debugging session. What we already found, running
cells one at a time by hand:

- The base model alone (no adapter), loaded full-precision in bf16, generates
  correct, coherent text. Twice, independently. **The shared generation code
  is not the bug.**
- M1's adapter attached to that same full-precision bf16 base — both as a
  live PeftModel wrapper *and* fully merged into the weights — produces
  degenerate repeated-character garbage. Since merging rules out a
  PEFT/caching bug, the problem is the **combination** of base weights and
  adapter delta.
- Leading hypothesis: M1's adapter was trained against a **4-bit NF4
  quantized** base (that's what `sft.py` uses), computed in **float16**. Its
  learned delta implicitly compensates for that quantization's specific
  error. Applying the same delta to the full-precision, unquantized,
  **bfloat16** base (what `generate.py` was doing) is a different arithmetic
  regime the delta was never calibrated for.

This notebook tests that hypothesis directly: load the base model **exactly
the way training did** (4-bit NF4, float16 compute) and generate with each
adapter. If that fixes it, `generate.py` gets a permanent fix to match; if
not, we're still in a real bug and this notebook's granular progress prints
will show where.

**Every model-loading cell explicitly frees the previous model's GPU memory
before loading the next one** (`del` + `torch.cuda.empty_cache()`). Running
three unfreed models back to back (base bf16 + PeftModel wrapper + merged
copy, all left resident) is the most likely reason a later cell hung for 30
minutes in the earlier ad-hoc version of this test — GPU memory pressure and
allocator fragmentation, not a genuine deadlock.

**Before running:** attach `verilog-slm-data`, `verilog-slm-m1-final`, and
`verilog-slm-m0-final` as inputs (right panel -> Add Data). Accelerator =
GPU, Internet = On. Run cells top to bottom -- each one prints where it is,
so if anything stalls past what its own history says is normal (noted in
each cell), interrupt that cell specifically rather than the whole session.

In [ ]:
# --- Bootstrap: repo + corpus + local copies of both adapters ---
import os, shutil, glob

REPO = "https://github.com/saiswaroop25-pixel/verilog-slm"
os.chdir('/kaggle/working')
if not os.path.exists('/kaggle/working/verilog-slm'):
    os.system(f'git clone {REPO} /kaggle/working/verilog-slm')
os.chdir('/kaggle/working/verilog-slm')
os.system('git pull')
os.makedirs('artifacts', exist_ok=True)

hits = glob.glob('/kaggle/input/**/corpus.jsonl', recursive=True)
if hits:
    shutil.copy(hits[0], 'artifacts/corpus.jsonl'); print('restored corpus <-', hits[0])
else:
    print('WARNING: corpus.jsonl not found -- attach verilog-slm-data')

def local_adapter(keyword, dest):
    cfgs = [p for p in glob.glob('/kaggle/input/**/adapter_config.json', recursive=True) if keyword in p]
    if not cfgs:
        print(f'WARNING: no adapter found with "{keyword}" in its path -- is that dataset attached?')
        return False
    src = os.path.dirname(sorted(cfgs, key=len)[0])
    shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree(src, dest)
    print(f'{dest} <- {src}')
    return True

have_m1 = local_adapter('m1-final', 'artifacts/m1_final')
have_m0 = local_adapter('m0-final', 'artifacts/m0_final')
print('done: have_m1 =', have_m1, '| have_m0 =', have_m0)

In [ ]:
!pip install -q -r requirements.txt -r requirements-train.txt
!pip install -q -U "torchao>=0.16.0"
print('installs done')

In [ ]:
import torch
print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available())
!nvidia-smi -L

In [ ]:
# Shared setup: tokenizer + a couple of real probe-split prompts + the
# generation helper. Nothing GPU-heavy in this cell.
import sys, json
sys.path.insert(0, '.')
from transformers import AutoTokenizer
from src.infer.generate import generate_batch
from src.utils.prompts import build_prompt

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.padding_side = "left"

probe_rows = [json.loads(l) for l in open('artifacts/corpus.jsonl') if '"split": "probe"' in l][:3]
prompts = [build_prompt(r['instruction']) for r in probe_rows]
print(f'loaded tokenizer and {len(prompts)} probe prompts')
for r in probe_rows:
    print(' -', r['instruction'][:100].replace(chr(10), ' '))

## Test 1: base model alone, full precision bf16 (sanity check)

Already confirmed working twice by hand. Re-running here for a clean,
complete record. Should take well under a minute once the weights are
cached from earlier in this session -- if this one hangs, something more
basic than the quantization hypothesis is wrong (network, cache, GPU
driver), and everything below is moot until that's fixed.

In [ ]:
import torch
from transformers import AutoModelForCausalLM

print('loading base (bf16, unquantized)...')
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="cuda:0", torch_dtype=torch.bfloat16)
base.eval()
print('loaded. generating...')
out = generate_batch(base, tok, prompts[:1], n=1, temperature=0.8, top_p=0.95, max_new_tokens=150)
print('=== base model alone (bf16) ===')
print(repr(out[0][0][:300]))

del base
import gc; gc.collect(); torch.cuda.empty_cache()
print('freed')

## Test 2: quantized base (matches training) + M1 adapter

This is the actual test of the hypothesis. Loading a 4-bit quantized model
means bitsandbytes quantizes the weights on the fly at load time (not just
a file read) -- expect roughly 1-3 minutes, not seconds. If this specific
cell passes 10 minutes with no further print after "attaching adapter...",
that's a real stall worth interrupting; if it's still on "loading quantized
base..." past 10 minutes, the quantization step itself is stuck.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

print('loading quantized base (4-bit NF4, float16 compute -- matches sft.py)...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
base_q = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="cuda:0", torch_dtype=torch.float16,
)
base_q.eval()
print('quantized base loaded. attaching M1 adapter...')

m1_q = PeftModel.from_pretrained(base_q, 'artifacts/m1_final')
m1_q.eval()
print('adapter attached. generating...')

out = generate_batch(m1_q, tok, prompts[:1], n=1, temperature=0.8, top_p=0.95, max_new_tokens=150)
print('=== quantized base + M1 adapter ===')
print(repr(out[0][0][:300]))

del base_q, m1_q
import gc; gc.collect(); torch.cuda.empty_cache()
print('freed')

## Test 3: quantized base (matches training) + M0 adapter

Same setup as Test 2, swapping in M0's adapter, for direct comparison. If
Test 2 came out readable, this tells us whether M0 is *also* fixed by
matching the quantization, or whether M0 has a separate, additional problem
on top (recall its earlier probe-split samples were garbled in a different
way than M1's -- varying gibberish vs. one repeated character).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

print('loading quantized base again (fresh instance)...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
base_q2 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="cuda:0", torch_dtype=torch.float16,
)
base_q2.eval()
print('quantized base loaded. attaching M0 adapter...')

m0_q = PeftModel.from_pretrained(base_q2, 'artifacts/m0_final')
m0_q.eval()
print('adapter attached. generating...')

out = generate_batch(m0_q, tok, prompts[:1], n=1, temperature=0.8, top_p=0.95, max_new_tokens=150)
print('=== quantized base + M0 adapter ===')
print(repr(out[0][0][:300]))

del base_q2, m0_q
import gc; gc.collect(); torch.cuda.empty_cache()
print('freed')

## How to read the three results

- **Test 1 readable, Test 2 readable** -> hypothesis confirmed: inference
  must match training's quantization. Fix `generate.py` permanently to load
  the base model 4-bit-quantized (same as `sft.py`'s `load_base_model_and_tokenizer`)
  instead of full precision, then redo the full diagnostic pass and M0-vs-M1
  evaluation against the corrected pipeline.
- **Test 1 readable, Test 2 still garbage** -> the quantization-mismatch
  hypothesis was wrong. Paste all three outputs and we dig further --
  likely something specific to how this adapter's weights were saved or
  how PEFT is loading them in this environment.
- **Test 3 readable like Test 2** -> M0 is fine too once matched to its
  training quantization; the earlier "M0 diagnostic" results are unreliable
  and need to be regenerated with the corrected pipeline before drawing any
  conclusion about M0's actual quality.
- **Test 2 readable but Test 3 still garbage** -> M0 has a second, separate
  problem on top of the quantization mismatch (consistent with its loss
  having been much worse than M1's, and the NaN instability it went
  through). M1 would be trustworthy; M0 would need retraining regardless.

Paste all three `===` printed blocks back into the conversation.